# Multicoordinate Wave Project

The multicoordinate wave example generates one BHaH project with several coordinate systems. It reuses the wave-equation right-hand side, reference metrics, precompute support, curvilinear boundary conditions, Method-of-Lines timestepping, and diagnostics in one project.

## Table of Contents

1. [Runtime context and generator discovery](#Runtime-context-and-generator-discovery)
2. [Generator constants](#Generator-constants)
3. [Project generation](#Project-generation)
4. [Generated file inventory](#Generated-file-inventory)
5. [Build and run](#Build-and-run)
6. [Related notebooks](#Related-notebooks)

## Runtime Context and Generator Discovery

Discovery uses `importlib.util.find_spec` so the packaged generator is not imported at notebook startup. Importing the generator module would execute generation-time code.

In [ ]:
import ast
import importlib.util
import subprocess
import sys
import tempfile
from pathlib import Path

import nrpy


print("Imported nrpy from:", nrpy.__file__)

generator_name = "nrpy.examples.wave_equation_multicoordinates"
generator_spec = importlib.util.find_spec(generator_name)
print("Generator discovered:", generator_spec is not None)

## Generator Constants

The generator source records the project name, coordinate systems, grid sizes, finite-difference order, radiation boundary order, Method-of-Lines method, and diagnostics choices. Reading those literal assignments is safe because it does not import or run the generator.

In [ ]:
tracked_names = {
    "project_name",
    "set_of_CoordSystems",
    "Nxx_dict",
    "MoL_method",
    "fd_order",
    "radiation_BC_fd_order",
    "enable_KreissOliger_dissipation",
    "boundary_conditions_desc",
}

generator_config = {}
if generator_spec is None or generator_spec.origin is None:
    raise RuntimeError(f"Generator not found: {generator_name}")
else:
    source = Path(generator_spec.origin).read_text(encoding="utf-8")
    tree = ast.parse(source)
    for node in tree.body:
        if isinstance(node, ast.Assign):
            for target in node.targets:
                if isinstance(target, ast.Name) and target.id in tracked_names:
                    try:
                        generator_config[target.id] = ast.literal_eval(node.value)
                    except (ValueError, SyntaxError):
                        pass

for name in sorted(generator_config):
    value = generator_config[name]
    if isinstance(value, set):
        value = sorted(value)
    print(name + ":", value)

## Project Generation

Generation runs inside a disposable workspace, not from the repository root or an active project directory. The subprocess `HOME` is also set to that workspace so user-level cache state is not needed.

In [ ]:
workspace_manager = tempfile.TemporaryDirectory(
    prefix=".nrpy-wave-multicoord-", dir=Path.cwd()
)
workspace_path = Path(workspace_manager.name)
project_path = workspace_path / "project" / "wave_equation_multicoordinates"

print("Temporary workspace:", workspace_path.name)
print("Project directory exists before generation:", project_path.exists())

generation_completed = False
if generator_spec is None:
    raise RuntimeError(f"Generator not found: {generator_name}")
else:
    generator_env = {"HOME": str(workspace_path)}
    generate = subprocess.run(
        [sys.executable, "-m", generator_name, "--disable_intrinsics"],
        cwd=workspace_path,
        env=generator_env,
        check=False,
        capture_output=True,
        text=True,
        timeout=600,
    )
    generation_completed = generate.returncode == 0
    print("Generator return code:", generate.returncode)
    if generate.stdout.strip():
        print("\n".join(generate.stdout.splitlines()[:16]))
    if generate.returncode != 0 and generate.stderr.strip():
        print("Generator stderr excerpt:")
        print("\n".join(generate.stderr.splitlines()[:16]))
    if not generation_completed:
        raise RuntimeError(
            "Multicoordinate generator failed; see stderr excerpt above."
        )

## Generated File Inventory

The generated project should contain a Makefile, a default parameter file, C-family source files, and generated headers. Coordinate-system evidence is summarized from the generator constants and generated files without printing absolute paths.

In [ ]:
if not generation_completed:
    print("Generated project inventory is unavailable because generation did not complete.")
elif not project_path.exists():
    print("Generated project directory is not present.")
else:
    required_paths = [
        project_path / "Makefile",
        project_path / "wave_equation_multicoordinates.par",
    ]
    for path in required_paths:
        print(path.relative_to(project_path), "exists:", path.exists())

    source_suffixes = {".c", ".cpp", ".cu"}
    header_suffixes = {".h", ".hpp", ".cuh"}
    generated_files = sorted(path for path in project_path.rglob("*") if path.is_file())
    source_files = [path for path in generated_files if path.suffix in source_suffixes]
    header_files = [path for path in generated_files if path.suffix in header_suffixes]

    print("Generated file count:", len(generated_files))
    print("C-family source count:", len(source_files))
    print("Header count:", len(header_files))
    print("First generated files:")
    for path in generated_files[:12]:
        print(" ", path.relative_to(project_path))

    coordinate_systems = sorted(generator_config.get("set_of_CoordSystems", []))
    print("Coordinate systems from generator constants:", coordinate_systems)
    coordinate_hits = {}
    for coord_system in coordinate_systems:
        hits = []
        for path in generated_files:
            relpath = path.relative_to(project_path)
            if coord_system in relpath.parts or f"__rfm__{coord_system}" in path.name:
                hits.append(str(relpath))
            if len(hits) >= 4:
                break
        coordinate_hits[coord_system] = hits

    print("Coordinate evidence:")
    for coord_system, hits in coordinate_hits.items():
        print(" ", coord_system + ":", hits)

## Build and Run

Building is optional for the notebook path because it depends on local compiler tools and runtime budget. From the generated project directory, run:

```bash
make -j2
./wave_equation_multicoordinates
```

## Related Notebooks

[Curvilinear wave equation](wave_equation_curvilinear.ipynb) introduces the single-coordinate curvilinear generator. [Curvilinear boundary conditions](../4-curvilinear/curvilinear_boundary_conditions.ipynb) explains the ghost-zone and parity calls used here.